In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [2]:
data_dir = '../../../data/processed/Misar-Stereo'

ad_mis13_rna = sc.read_h5ad(join(data_dir, 'Misar-E13/ad_rna.h5ad'))
ad_mis13_atac = sc.read_h5ad(join(data_dir, 'Misar-E13/ad_atac.h5ad'))

ad_mis15_rna = sc.read_h5ad(join(data_dir, 'Misar-E15/ad_rna.h5ad'))
ad_mis15_atac = sc.read_h5ad(join(data_dir, 'Misar-E15/ad_atac.h5ad'))

ad_mis18_rna = sc.read_h5ad(join(data_dir, 'Misar-E18/ad_rna.h5ad'))
ad_mis18_atac = sc.read_h5ad(join(data_dir, 'Misar-E18/ad_atac.h5ad'))

ad_ste12_rna = sc.read_h5ad(join(data_dir, 'Stereo-E12/ad_rna.h5ad'))
ad_ste14_rna = sc.read_h5ad(join(data_dir, 'Stereo-E14/ad_rna.h5ad')) 
ad_ste16_rna = sc.read_h5ad(join(data_dir, 'Stereo-E16/ad_rna.h5ad')) 

input_dict = { 
    'rna':   [ad_mis13_rna, ad_mis15_rna,  ad_mis18_rna,  ad_ste12_rna, ad_ste14_rna, ad_ste16_rna],
    'atac':  [ad_mis13_atac,ad_mis15_atac, ad_mis18_atac, None, None, None],
}

input_key = 'dimred_bc'
batch_key = 'Sample'

In [3]:
cache_dir = './cache_dir/Misar-Stereo'
df_cca = pd.read_csv(join(cache_dir, 'cca_coordinates.csv'), index_col=0)    # exported using seurat integration pipeline 
for adx in input_dict['rna']:
    adx.obsm[input_key] = df_cca.loc[adx.obs_names].values

Epigenome_preprocess(input_dict['atac'], 
                     batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)   

Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).


In [4]:
def stack(xl, key):
    xs, ns = [], []
    for adx in xl:
        if adx is not None:
            xs.append(adx.obsm[key])
            ns.append(adx.obs_names)
    df = pd.DataFrame(np.vstack(xs), index=np.hstack(ns))
    return df

for m1, m2 in zip(['rna', 'atac'], ['RNA', 'ATAC']):
    fig_dir = f'../../../results/embeddings/Leiden-{m2}/Misar-Stereo'
    os.makedirs(fig_dir, exist_ok=True)
    df = stack(input_dict[m1], input_key)
    df.to_csv(join(fig_dir, 'df_emb.csv'))